# MyGPT2 — Checkpoint Evaluation

This notebook evaluates a trained MyGPT2 checkpoint without changing the training pipeline.

It covers:
- checkpoint loading and integrity checks
- text generation with temperature / top-k / top-p sampling
- fixed-prompt qualitative testing
- sampling comparison
- training-loss → perplexity conversion
- optional validation-loss evaluation
- GPU memory reporting

Default checkpoint: `artifacts/checkpoints/final_step_00010000.pt`


In [1]:
# ============================================================
# 1. Imports and project paths
# ============================================================

from __future__ import annotations

import math
import random
import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd()

# Support both:
#   D:\Gpt2_v01
# and:
#   D:\Gpt2_v01\evaluation\notebooks
if not (PROJECT_ROOT / "model").exists():
    candidates = list(PROJECT_ROOT.parents)
    for candidate in candidates:
        if (candidate / "model").exists() and (candidate / "tokenizer").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer
from training.checkpoint import load_checkpoint

print("Project Root:", PROJECT_ROOT)


Project Root: d:\Gpt2_v01


In [2]:
# ============================================================
# 2. Evaluation configuration
# ============================================================

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "checkpoints"
    / "final_step_00010000.pt"
)

TOKENIZER_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tokenizer"
    / "tokenizer.json"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

TEMPERATURE = 0.8
TOP_K = 50
TOP_P = 0.95
MAX_NEW_TOKENS = 80

PROMPTS = [
    "Once upon a time",
    "Sam wanted to play",
    "my name is harsh prabhakar",
    "i didn't wanted to build this modle ",
    "The little girl went",
    "i didn't wanted to build this modle ",
]

print("Checkpoint:", CHECKPOINT_PATH)
print("Tokenizer :", TOKENIZER_PATH)
print("Device    :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU       :", torch.cuda.get_device_name(0))


Checkpoint: d:\Gpt2_v01\artifacts\checkpoints\final_step_00010000.pt
Tokenizer : d:\Gpt2_v01\artifacts\tokenizer\tokenizer.json
Device    : cuda
GPU       : NVIDIA GeForce RTX 5060 Ti


In [3]:
# ============================================================
# 3. Seed and file checks
# ============================================================

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT_PATH}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found:\n{TOKENIZER_PATH}"
    )

checkpoint_size_mb = CHECKPOINT_PATH.stat().st_size / (1024 ** 2)
tokenizer_size_mb = TOKENIZER_PATH.stat().st_size / (1024 ** 2)

print(f"Checkpoint size : {checkpoint_size_mb:.2f} MB")
print(f"Tokenizer size  : {tokenizer_size_mb:.2f} MB")
print("File checks     : PASSED")


Checkpoint size : 1259.35 MB
Tokenizer size  : 2.16 MB
File checks     : PASSED


## 4. Load the 10k checkpoint

A fresh model is constructed with the same `GPTConfig`, then the saved model weights are loaded from `final_step_00010000.pt`.


In [4]:
# ============================================================
# 4. Load tokenizer, model and checkpoint
# ============================================================

config = GPTConfig()

tokenizer = MyGPTTokenizer.load(TOKENIZER_PATH)

if tokenizer.vocabulary_size != config.vocab_size:
    raise RuntimeError(
        "Vocabulary mismatch.\n"
        f"Tokenizer: {tokenizer.vocabulary_size}\n"
        f"Config:    {config.vocab_size}"
    )

model = MyGPTModel(config).to(DEVICE)
model.eval()

checkpoint = load_checkpoint(
    path=CHECKPOINT_PATH,
    model=model,
    optimizer=None,
    scheduler=None,
    device=DEVICE,
    restore_rng=False,
)

print("Checkpoint version:", checkpoint.get("checkpoint_version"))
print("Saved epoch      :", checkpoint.get("epoch"))
print("Saved global step:", checkpoint.get("global_step"))
print("Saved train loss :", checkpoint.get("train_loss"))
print("Saved val loss   :", checkpoint.get("val_loss"))
print("Model loaded     : PASSED")


Checkpoint version: 1.2
Saved epoch      : 0
Saved global step: 10000
Saved train loss : 1.3898952007293701
Saved val loss   : None
Model loaded     : PASSED


In [5]:
# ============================================================
# 5. Model summary and checkpoint sanity checks
# ============================================================

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Vocabulary size : {tokenizer.vocabulary_size:,}")
print(f"Context length  : {config.max_position_embeddings}")
print(f"Hidden size     : {config.hidden_size}")
print(f"Layers          : {config.num_layers}")
print(f"Attention heads : {config.num_attention_heads}")
print(f"Total params    : {total_params:,} ({total_params / 1e6:.2f}M)")
print(f"Trainable params: {trainable_params:,}")

assert checkpoint.get("global_step") == 10000, (
    "Expected the 10k checkpoint, but the saved global step "
    f"is {checkpoint.get('global_step')}."
)

print("Checkpoint step : PASSED")


Vocabulary size : 32,000
Context length  : 512
Hidden size     : 768
Layers          : 12
Attention heads : 12
Total params    : 110,025,216 (110.03M)
Trainable params: 110,025,216
Checkpoint step : PASSED


## 6. Sampling utilities

The generation code uses the same model forward API as the current project and supports temperature, top-k, and top-p sampling.


In [6]:
# ============================================================
# 6. Sampling helper
# ============================================================

def sample_next_token(
    logits: torch.Tensor,
    *,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
) -> int:
    if temperature <= 0:
        raise ValueError("temperature must be > 0.")

    if not 0 < top_p <= 1:
        raise ValueError("top_p must be in (0, 1].")

    logits = logits.float() / temperature

    # ---------------------------
    # Top-k
    # ---------------------------
    if top_k > 0:
        top_k = min(top_k, logits.shape[-1])

        values, _ = torch.topk(
            logits,
            top_k,
        )

        cutoff = values[-1]

        logits = torch.where(
            logits < cutoff,
            torch.full_like(logits, float("-inf")),
            logits,
        )

    # ---------------------------
    # Top-p
    # ---------------------------
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(
            logits,
            descending=True,
        )

        sorted_probs = torch.softmax(
            sorted_logits,
            dim=-1,
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1,
        )

        remove_mask = cumulative_probs > top_p

        if remove_mask.numel() > 1:
            remove_mask[1:] = remove_mask[:-1].clone()

        remove_mask[0] = False

        sorted_logits = sorted_logits.masked_fill(
            remove_mask,
            float("-inf"),
        )

        filtered_logits = torch.full_like(
            logits,
            float("-inf"),
        )

        filtered_logits.scatter_(
            0,
            sorted_indices,
            sorted_logits,
        )

        logits = filtered_logits

    probs = torch.softmax(
        logits,
        dim=-1,
    )

    if not torch.isfinite(probs).all():
        raise RuntimeError(
            "Sampling probabilities are NaN or infinite."
        )

    return int(
        torch.multinomial(
            probs,
            num_samples=1,
        ).item()
    )


In [7]:
# ============================================================
# 7. Text generation
# ============================================================

@torch.inference_mode()
def generate_text(
    prompt: str,
    *,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_k: int = TOP_K,
    top_p: float = TOP_P,
) -> tuple[str, list[int]]:
    model.eval()

    prompt_ids = tokenizer.encode(prompt)

    if not prompt_ids:
        raise ValueError(
            f"Prompt produced no tokens: {prompt!r}"
        )

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    context_limit = int(
        config.max_position_embeddings
    )

    eos_id = getattr(
        config,
        "eos_token_id",
        None,
    )

    for _ in range(max_new_tokens):

        if input_ids.shape[1] > context_limit:
            input_ids = input_ids[:, -context_limit:]

        output = model(
            input_ids=input_ids
        )

        if isinstance(output, tuple):
            logits = output[0]
        elif hasattr(output, "logits"):
            logits = output.logits
        else:
            raise RuntimeError(
                "Model output does not contain logits."
            )

        next_token_id = sample_next_token(
            logits[0, -1],
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        next_token = torch.tensor(
            [[next_token_id]],
            dtype=torch.long,
            device=DEVICE,
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1,
        )

        if (
            eos_id is not None
            and next_token_id == eos_id
        ):
            break

    ids = (
        input_ids[0]
        .detach()
        .cpu()
        .tolist()
    )

    text = tokenizer.decode(ids)

    return text, ids


## 8. Fixed-prompt response test

This is the first qualitative evaluation of the 10k checkpoint.

Look for:
- coherent words and sentences
- prompt relevance
- TinyStories-like narrative structure
- low repetition
- reasonable termination
- absence of obvious tokenization artifacts


In [8]:
# ============================================================
# 8. Generate responses for fixed prompts
# ============================================================

generation_results = []

for index, prompt in enumerate(PROMPTS, start=1):

    text, ids = generate_text(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
    )

    prompt_token_count = len(
        tokenizer.encode(prompt)
    )

    generated_token_count = max(
        0,
        len(ids) - prompt_token_count,
    )

    result = {
        "prompt": prompt,
        "text": text,
        "generated_tokens": generated_token_count,
    }

    generation_results.append(result)

    print("=" * 75)
    print(f"Prompt {index}: {prompt}")
    print("-" * 75)
    print(text)
    print()
    print(
        f"Generated tokens: {generated_token_count}"
    )


Prompt 1: Once upon a time
---------------------------------------------------------------------------
 Once upon a time was a girl who liked to play in the park. She had a lot of friends, but she did not like to share her toys with others. She said, "These are my toys. Go away, these are my food."

One day, her mother told her that they had to go to the doctor. The doctor was nice, but smiled and said, "I will be back

Generated tokens: 80
Prompt 2: Sam wanted to play
---------------------------------------------------------------------------
 Sam wanted to play with his toy gun, but he was afraid of what his dad said. He looked around the house for something fun to do. He saw a big box of old clothes and ran to the door. He opened the box and saw a lot of shiny things. He thought they were pretty and wanted to try them on.

He pushed the door open and saw a lot of clothes and shoes.

Generated tokens: 80
Prompt 3: my name is harsh prabhakar
-------------------------------------------

## 9. Sampling comparison

Run the same prompt at several temperatures. Lower temperature generally makes sampling more conservative; higher temperature increases variation.


In [9]:
# ============================================================
# 9. Temperature comparison
# ============================================================

comparison_prompt = "Once upon a time"

for temperature in [0.6, 0.8, 1.0]:

    random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    text, _ = generate_text(
        comparison_prompt,
        max_new_tokens=60,
        temperature=temperature,
        top_k=TOP_K,
        top_p=TOP_P,
    )

    print("=" * 75)
    print(f"Temperature = {temperature}")
    print("-" * 75)
    print(text)


Temperature = 0.6
---------------------------------------------------------------------------
 Once upon a time was a girl who liked to play with her toys. She had a lot of toys, but she did not like to share them with anyone. She always wanted to have everything for her.

One day, she saw a big truck parked in the park. It was shiny and red and had
Temperature = 0.8
---------------------------------------------------------------------------
 Once upon a time was a girl who liked to play in the park. She had a lot of friends, but she did not like to share her toys with others. She said, "These are my toys. Go away, these are my food."

One day, her mother told her that they had to
Temperature = 1.0
---------------------------------------------------------------------------
 Once upon a time was a girl who liked to play in the park. She had a lot of friends, but she did not have enough money. She wanted to go to the park and see the birds, the flowers and the birds.

One day, she saw a 

## 10. Training loss → perplexity

For cross-entropy language-model loss:

`perplexity = exp(loss)`

This notebook reports the saved checkpoint loss as a baseline. A validation loss should eventually become the main generalization metric.


In [10]:
# ============================================================
# 10. Perplexity from saved loss
# ============================================================

saved_train_loss = checkpoint.get("train_loss")
saved_val_loss = checkpoint.get("val_loss")

if (
    saved_train_loss is not None
    and math.isfinite(float(saved_train_loss))
):
    train_perplexity = math.exp(
        float(saved_train_loss)
    )

    print(
        f"Saved train loss : "
        f"{float(saved_train_loss):.6f}"
    )

    print(
        f"Train perplexity : "
        f"{train_perplexity:.4f}"
    )

else:
    print(
        "Saved train loss is unavailable."
    )

if (
    saved_val_loss is not None
    and math.isfinite(float(saved_val_loss))
):
    val_perplexity = math.exp(
        float(saved_val_loss)
    )

    print(
        f"Saved val loss   : "
        f"{float(saved_val_loss):.6f}"
    )

    print(
        f"Val perplexity   : "
        f"{val_perplexity:.4f}"
    )

else:
    print(
        "Saved validation loss is unavailable."
    )


Saved train loss : 1.389895
Train perplexity : 4.0144
Saved validation loss is unavailable.


## 11. Optional validation evaluation

The current 10k run did not establish a dedicated official validation split. This cell is intentionally disabled by default.

Before enabling it, define the validation dataset/split you want to use as the project's official evaluation set.


In [11]:
# ============================================================
# 11. Optional validation evaluation
# ============================================================

ENABLE_VALIDATION = False

if ENABLE_VALIDATION:

    from training.dataloader import create_dataloader

    VAL_BATCH_SIZE = config.batch_size
    VAL_SEQUENCE_LENGTH = (
        config.max_position_embeddings
    )

    # Set this according to the project's official
    # validation-data policy.
    VAL_MAX_DOCUMENTS = 10000
    MAX_VAL_BATCHES = 100

    val_loader = create_dataloader(
        tokenizer=tokenizer,
        sequence_length=VAL_SEQUENCE_LENGTH,
        batch_size=VAL_BATCH_SIZE,
        max_documents=VAL_MAX_DOCUMENTS,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
    )

    model.eval()

    total_loss = 0.0
    batches = 0

    with torch.inference_mode():

        for batch in val_loader:

            input_ids, labels = batch

            input_ids = input_ids.to(
                DEVICE,
                non_blocking=True,
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True,
            )

            output = model(
                input_ids=input_ids,
                labels=labels,
            )

            if isinstance(output, tuple):
                loss = output[1]
            else:
                loss = output.loss

            total_loss += float(
                loss.item()
            )

            batches += 1

            if batches >= MAX_VAL_BATCHES:
                break

    if batches == 0:
        raise RuntimeError(
            "Validation loader produced zero batches."
        )

    validation_loss = (
        total_loss / batches
    )

    print(
        f"Validation loss : "
        f"{validation_loss:.6f}"
    )

    print(
        f"Validation ppl  : "
        f"{math.exp(validation_loss):.4f}"
    )


In [12]:
# ============================================================
# 12. Final evaluation summary
# ============================================================

print("=" * 75)
print("Evaluation Summary")
print("=" * 75)

print(
    f"Checkpoint step : "
    f"{checkpoint.get('global_step')}"
)

print(
    f"Training loss   : "
    f"{checkpoint.get('train_loss')}"
)

print(
    f"Prompts tested  : "
    f"{len(PROMPTS)}"
)

if DEVICE.type == "cuda":

    allocated = (
        torch.cuda.memory_allocated(DEVICE)
        / (1024 ** 2)
    )

    reserved = (
        torch.cuda.memory_reserved(DEVICE)
        / (1024 ** 2)
    )

    print(
        f"GPU allocated   : "
        f"{allocated:.2f} MB"
    )

    print(
        f"GPU reserved    : "
        f"{reserved:.2f} MB"
    )

print(
    "Generation test : ✅ COMPLETED"
)

print("=" * 75)


Evaluation Summary
Checkpoint step : 10000
Training loss   : 1.3898952007293701
Prompts tested  : 6
GPU allocated   : 452.46 MB
GPU reserved    : 532.00 MB
Generation test : ✅ COMPLETED


## Interpretation checklist

Before starting full training, manually inspect the generated responses.

**Good signs**
- sentences are mostly grammatical
- continuations remain related to the prompt
- short stories have characters/actions/events
- repeated phrases are limited
- output does not collapse into token noise

**Warning signs**
- repeated token/phrase loops
- unrelated continuations
- malformed words throughout
- output that immediately becomes incoherent
- strong dependence on one prompt style

The 10k training loss is a useful baseline, but generation quality and a real validation loss are the next decision points.
